<a href="https://colab.research.google.com/github/HemanthM17/Gen-Ai-Assignment/blob/main/task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏷️ Fine-Tuning DistilBERT for POS Tagging & Chunking
**Data Science Internship — February 2026**  
**Assignment NLP-5: Token Classification — POS Tagging & Chunking**  
**Innomatics Research Labs**

---
### Pipeline Flow
```
Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison
```

### Dataset
**CoNLL-2003** — Loaded directly via Hugging Face `datasets`. Contains POS tags and Chunk tags.  
No manual download needed.

### Model
**distilbert-base-uncased** — A lighter, faster version of BERT (40% smaller, 60% faster, retains 97% of BERT performance).

---
## 📦 Step 0: Install & Import Libraries

In [1]:
!pip install -q transformers "datasets>=2.14" seqeval torch evaluate
!pip install -q -U datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 10.6 MB/s eta 0:00:00


In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Libraries imported.')
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')

✅ Libraries imported.
Device  : cpu
PyTorch : 2.10.0+cpu


---
## 📂 Task 1: Dataset Selection

**Dataset: CoNLL-2003**

The CoNLL-2003 dataset is a standard NLP benchmark for sequence labelling tasks. It contains:
- **POS tags** — Part-of-Speech labels (NN, VB, JJ, etc.)
- **Chunk tags** — Phrase-level labels in IOB format (B-NP, I-NP, B-VP, etc.)
- **NER tags** — Named Entity Recognition labels (not used in this task)

Each sample is a tokenized sentence with one label per word.

In [4]:
print('Loading CoNLL-2003 dataset...')
from datasets import load_dataset

dataset = load_dataset(
    'parquet',
    data_files={
        'train'     : 'https://huggingface.co/datasets/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003/train/0000.parquet',
        'validation': 'https://huggingface.co/datasets/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003/validation/0000.parquet',
        'test'      : 'https://huggingface.co/datasets/conll2003/resolve/refs%2Fconvert%2Fparquet/conll2003/test/0000.parquet',
    }
)

print('\n✅ Dataset loaded.')
print(f'Splits available  : {list(dataset.keys())}')
print(f'Training samples  : {len(dataset["train"])}')
print(f'Validation samples: {len(dataset["validation"])}')
print(f'Test samples      : {len(dataset["test"])}')
print(f'\nFeatures: {dataset["train"].features}')

Loading CoNLL-2003 dataset...


conll2003/train/0000.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]


✅ Dataset loaded.
Splits available  : ['train', 'validation', 'test']
Training samples  : 14041
Validation samples: 3250
Test samples      : 3453

Features: {'id': Value('string'), 'tokens': List(Value('string')), 'pos_tags': List(ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'])), 'chunk_tags': List(ClassLabel(names=['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'])), 'ner_tags': List(ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']))}


In [5]:
# Extract label lists
# Parquet-loaded datasets may store tags as plain int sequences.
# We define the standard CoNLL-2003 label lists explicitly — they never change.

pos_label_list = [
    '""', 'NN', 'NNS', 'NNP', 'NNPS', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ',
    'JJ', 'JJR', 'JJS', 'RB', 'RBR', 'RBS', 'DT', 'IN', 'CC', 'CD', 'MD', 'PRP',
    'PRP$', 'WP', 'WP$', 'WRB', 'WDT', 'TO', 'UH', 'RP', 'PDT', 'EX', 'FW', 'LS',
    'POS', 'SYM', '``', "''", ',', '.', ':', '$', '#', '-LRB-', '-RRB-'
]

chunk_label_list = [
    'O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP',
    'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP',
    'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'
]

# Try to get from dataset features first (works if Sequence/ClassLabel is preserved)
try:
    pos_label_list   = dataset['train'].features['pos_tags'].feature.names
    chunk_label_list = dataset['train'].features['chunk_tags'].feature.names
    print('Labels loaded from dataset features.')
except Exception:
    print('Labels loaded from hardcoded CoNLL-2003 standard list.')

print(f'POS Tag labels   ({len(pos_label_list)}) : {pos_label_list}')
print(f'\nChunk Tag labels ({len(chunk_label_list)}): {chunk_label_list}')

# Sample sentence
sample = dataset['train'][0]
print('\n--- Sample Sentence ---')
print(f'Words      : {sample["tokens"]}')
print(f'POS tags   : {[pos_label_list[t] for t in sample["pos_tags"]]}')
print(f'Chunk tags : {[chunk_label_list[t] for t in sample["chunk_tags"]]}')


Labels loaded from dataset features.
POS Tag labels   (47) : ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']

Chunk Tag labels (23): ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']

--- Sample Sentence ---
Words      : ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS tags   : ['NNP', 'VBZ', 'JJ', 'NN', 'TO', 'VB', 'JJ', 'NN', '.']
Chunk tags : ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O']


In [ ]:
# Visualise label distributions
all_pos_tags   = [pos_label_list[t]   for ex in dataset['train'] for t in ex['pos_tags']]
all_chunk_tags = [chunk_label_list[t] for ex in dataset['train'] for t in ex['chunk_tags']]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

pos_counts   = Counter(all_pos_tags)
chunk_counts = Counter(all_chunk_tags)

# POS distribution
axes[0].bar(pos_counts.keys(), pos_counts.values(), color='#3b82f6', edgecolor='white')
axes[0].set_title('POS Tag Distribution (Training Set)', fontweight='bold')
axes[0].set_xlabel('POS Tag')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=70)

# Chunk distribution
axes[1].bar(chunk_counts.keys(), chunk_counts.values(), color='#10b981', edgecolor='white')
axes[1].set_title('Chunk Tag Distribution (Training Set)', fontweight='bold')
axes[1].set_xlabel('Chunk Tag')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=70)

plt.tight_layout()
plt.show()

print(f'Total tokens in training set: {len(all_pos_tags):,}')


##  Task 2: Data Preprocessing & Label Alignment

In [7]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LEN    = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

# Demonstrate the subword problem
words   = ['John', 'works', 'at', 'Google', 'in', 'California']
encoded = tokenizer(words, is_split_into_words=True)
print(f'\nWords     : {words}')
print(f'Token IDs : {encoded["input_ids"]}')
print(f'Tokens    : {tokenizer.convert_ids_to_tokens(encoded["input_ids"])}')
print(f'Word IDs  : {encoded.word_ids()}')
print('\nword_ids() maps each token back to its original word index.')
print('None = special token ([CLS]/[SEP])')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased

Words     : ['John', 'works', 'at', 'Google', 'in', 'California']
Token IDs : [101, 2198, 2573, 2012, 8224, 1999, 2662, 102]
Tokens    : ['[CLS]', 'john', 'works', 'at', 'google', 'in', 'california', '[SEP]']
Word IDs  : [None, 0, 1, 2, 3, 4, 5, None]

word_ids() maps each token back to its original word index.
None = special token ([CLS]/[SEP])


In [8]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenize words and align labels with subword tokens.

    For each word:
      - First subword token  → gets the word's original label
      - Subsequent subwords  → get -100 (ignored in loss)
      - Special tokens       → get -100

    Args:
        examples     : batch of dataset examples
        label_column : 'pos_tags' or 'chunk_tags'
    """
    tokenized = tokenizer(
        examples['tokens'],
        truncation         = True,
        max_length         = MAX_LEN,
        is_split_into_words = True
    )

    all_labels = []
    for i, labels in enumerate(examples[label_column]):
        word_ids        = tokenized.word_ids(batch_index=i)
        previous_word_id = None
        label_ids        = []

        for word_id in word_ids:
            if word_id is None:
                # Special token [CLS] or [SEP] → ignore
                label_ids.append(-100)
            elif word_id != previous_word_id:
                # First subword of a new word → assign real label
                label_ids.append(labels[word_id])
            else:
                # Continuation subword → ignore
                label_ids.append(-100)

            previous_word_id = word_id

        all_labels.append(label_ids)

    tokenized['labels'] = all_labels
    return tokenized


print('✅ tokenize_and_align_labels() defined.')

# Demonstrate alignment
sample_ex = dataset['train'][0]
sample_tokens = sample_ex['tokens']
sample_pos    = sample_ex['pos_tags']

encoded_sample = tokenizer(sample_tokens, is_split_into_words=True)
subword_tokens = tokenizer.convert_ids_to_tokens(encoded_sample['input_ids'])
word_ids_sample = encoded_sample.word_ids()

prev = None
aligned_labels = []
for wid in word_ids_sample:
    if wid is None:
        aligned_labels.append('IGN')
    elif wid != prev:
        aligned_labels.append(pos_label_list[sample_pos[wid]])
    else:
        aligned_labels.append('IGN')
    prev = wid

df_demo = pd.DataFrame({'Token': subword_tokens, 'Label': aligned_labels})
print('\nLabel Alignment Demo (first 15 tokens):')
print(df_demo.head(15).to_string(index=False))

✅ tokenize_and_align_labels() defined.

Label Alignment Demo (first 15 tokens):
  Token Label
  [CLS]   IGN
     eu   NNP
rejects   VBZ
 german    JJ
   call    NN
     to    TO
boycott    VB
british    JJ
   lamb    NN
      .     .
  [SEP]   IGN


In [9]:
# Tokenize and align for POS task
print('Tokenizing dataset for POS Tagging...')
pos_tokenized = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, 'pos_tags'),
    batched=True,
    remove_columns=dataset['train'].column_names
)

# Tokenize and align for Chunking task
print('Tokenizing dataset for Chunking...')
chunk_tokenized = dataset.map(
    lambda ex: tokenize_and_align_labels(ex, 'chunk_tags'),
    batched=True,
    remove_columns=dataset['train'].column_names
)

print('\n✅ Both tasks tokenized and labels aligned.')
print(f'POS tokenized features   : {list(pos_tokenized["train"].features.keys())}')
print(f'Chunk tokenized features : {list(chunk_tokenized["train"].features.keys())}')

Tokenizing dataset for POS Tagging...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Tokenizing dataset for Chunking...


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


✅ Both tasks tokenized and labels aligned.
POS tokenized features   : ['input_ids', 'token_type_ids', 'attention_mask', 'labels']
Chunk tokenized features : ['input_ids', 'token_type_ids', 'attention_mask', 'labels']



##  Task 3: Model Setup

In [10]:
def build_model(label_list, model_name=MODEL_NAME):
    """
    Build AutoModelForTokenClassification with correct label mappings.

    Args:
        label_list : list of label strings
        model_name : HuggingFace model identifier

    Returns:
        model, id2label, label2id
    """
    id2label  = {i: label for i, label in enumerate(label_list)}
    label2id  = {label: i for i, label in enumerate(label_list)}

    model = AutoModelForTokenClassification.from_pretrained(
        model_name,
        num_labels = len(label_list),
        id2label   = id2label,
        label2id   = label2id
    )
    return model, id2label, label2id


# Build POS model
print('Building POS Tagging model...')
pos_model, pos_id2label, pos_label2id = build_model(pos_label_list)
print(f'✅ POS model ready. Labels: {len(pos_label_list)}')

# Build Chunking model
print('\nBuilding Chunking model...')
chunk_model, chunk_id2label, chunk_label2id = build_model(chunk_label_list)
print(f'✅ Chunk model ready. Labels: {len(chunk_label_list)}')

# Model info
total_params = sum(p.numel() for p in pos_model.parameters())
print(f'\nModel     : {MODEL_NAME}')
print(f'Parameters: {total_params:,}')

Building POS Tagging model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ POS model ready. Labels: 47

Building Chunking model...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Chunk model ready. Labels: 23

Model     : distilbert-base-uncased
Parameters: 66,399,023



## Task 4: Training


In [11]:
from seqeval.metrics import (
    precision_score as seq_precision,
    recall_score    as seq_recall,
    f1_score        as seq_f1,
    classification_report as seq_report
)

def make_compute_metrics(id2label):
    """
    Returns a compute_metrics function for the Trainer.
    Uses seqeval for token-level sequence evaluation.
    Ignores -100 labels (subwords and special tokens).
    """
    def compute_metrics(eval_preds):
        logits, labels = eval_preds
        predictions    = np.argmax(logits, axis=-1)

        true_labels = [
            [id2label[l] for l in label_row if l != -100]
            for label_row in labels
        ]
        true_preds = [
            [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
            for pred_row, label_row in zip(predictions, labels)
        ]

        return {
            'precision': seq_precision(true_labels, true_preds),
            'recall':    seq_recall(true_labels, true_preds),
            'f1':        seq_f1(true_labels, true_preds),
        }
    return compute_metrics


data_collator = DataCollatorForTokenClassification(tokenizer)
print('✅ Metrics and data collator ready.')

✅ Metrics and data collator ready.


In [ ]:
# ── TRAIN POS TAGGING MODEL ──────────────────────────────────────────────────
print('=' * 60)
print('TRAINING: POS Tagging Model')
print('=' * 60)

pos_training_args = TrainingArguments(
    output_dir              = './pos_model',
    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    learning_rate           = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs        = 3,
    weight_decay            = 0.01,
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    logging_steps           = 100,
    report_to               = 'none',
)

pos_trainer = Trainer(
    model           = pos_model,
    args            = pos_training_args,
    train_dataset   = pos_tokenized['train'],
    eval_dataset    = pos_tokenized['validation'],
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(pos_id2label)
)

print('Starting POS training...')
pos_train_result = pos_trainer.train()
print('\n✅ POS Tagging model training complete.')
print(f'Training loss : {pos_train_result.training_loss:.4f}')

In [ ]:
# ── TRAIN CHUNKING MODEL ─────────────────────────────────────────────────────
print('=' * 60)
print('TRAINING: Chunking Model')
print('=' * 60)

chunk_training_args = TrainingArguments(
    output_dir              = './chunk_model',
    eval_strategy           = 'epoch',
    save_strategy           = 'epoch',
    learning_rate           = 2e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 16,
    num_train_epochs        = 3,
    weight_decay            = 0.01,
    load_best_model_at_end  = True,
    metric_for_best_model   = 'f1',
    logging_steps           = 100,
    report_to               = 'none',
)

chunk_trainer = Trainer(
    model           = chunk_model,
    args            = chunk_training_args,
    train_dataset   = chunk_tokenized['train'],
    eval_dataset    = chunk_tokenized['validation'],
    data_collator   = data_collator,
    compute_metrics = make_compute_metrics(chunk_id2label)
)

print('Starting Chunking training...')
chunk_train_result = chunk_trainer.train()
print('\n✅ Chunking model training complete.')
print(f'Training loss : {chunk_train_result.training_loss:.4f}')


##  Task 5: Evaluation using seqeval

In [ ]:
def full_evaluate(trainer, tokenized_data, id2label, task_name):
    """
    Run full evaluation on test set and print detailed report.
    Returns predictions and true labels.
    """
    print(f'\n{"="*60}')
    print(f'EVALUATION: {task_name}')
    print(f'{"="*60}')

    predictions_output = trainer.predict(tokenized_data['test'])
    logits  = predictions_output.predictions
    labels  = predictions_output.label_ids
    preds   = np.argmax(logits, axis=-1)

    true_labels = [
        [id2label[l] for l in row if l != -100]
        for row in labels
    ]
    true_preds = [
        [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(preds, labels)
    ]

    prec = seq_precision(true_labels, true_preds)
    rec  = seq_recall(true_labels, true_preds)
    f1   = seq_f1(true_labels, true_preds)

    print(f'Precision : {prec:.4f} ({prec*100:.2f}%)')
    print(f'Recall    : {rec:.4f}  ({rec*100:.2f}%)')
    print(f'F1 Score  : {f1:.4f}  ({f1*100:.2f}%)')
    print('\nDetailed Classification Report:')
    print(seq_report(true_labels, true_preds))

    return true_preds, true_labels, {'precision': prec, 'recall': rec, 'f1': f1}


# Evaluate POS model
pos_preds, pos_true, pos_metrics = full_evaluate(
    pos_trainer, pos_tokenized, pos_id2label, 'POS Tagging'
)

# Evaluate Chunking model
chunk_preds, chunk_true, chunk_metrics = full_evaluate(
    chunk_trainer, chunk_tokenized, chunk_id2label, 'Chunking'
)

In [ ]:
# Visualise metrics comparison
metrics_names = ['Precision', 'Recall', 'F1 Score']
pos_vals   = [pos_metrics['precision'],   pos_metrics['recall'],   pos_metrics['f1']]
chunk_vals = [chunk_metrics['precision'], chunk_metrics['recall'], chunk_metrics['f1']]

x     = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, pos_vals,   width, label='POS Tagging', color='#3b82f6', edgecolor='white')
bars2 = ax.bar(x + width/2, chunk_vals, width, label='Chunking',    color='#10b981', edgecolor='white')

ax.bar_label(bars1, fmt='%.3f', padding=3, fontsize=10)
ax.bar_label(bars2, fmt='%.3f', padding=3, fontsize=10)

ax.set_title('POS Tagging vs Chunking — Evaluation Metrics', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


##  Task 6: Inference on Custom Sentences

In [ ]:
def predict_tags(sentence, model, tokenizer, id2label):
    """
    Predict POS or Chunk tags for a raw sentence.

    Steps:
      1. Split sentence into words
      2. Tokenize with subword handling
      3. Run model forward pass
      4. Take argmax of logits to get predicted label IDs
      5. Map back to label strings, keeping only first subword per word

    Returns:
        List of (word, predicted_tag) tuples
    """
    model.eval()
    words   = sentence.split()
    encoded = tokenizer(
        words,
        is_split_into_words = True,
        return_tensors      = 'pt',
        truncation          = True,
        max_length          = MAX_LEN
    ).to(device)

    model = model.to(device)

    with torch.no_grad():
        outputs = model(**encoded)

    logits   = outputs.logits
    pred_ids = torch.argmax(logits, dim=-1).squeeze().tolist()
    word_ids = encoded.word_ids()

    results  = []
    seen_ids = set()

    for token_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen_ids:
            continue
        seen_ids.add(word_id)
        tag = id2label[pred_ids[token_idx]]
        results.append((words[word_id], tag))

    return results


def display_predictions(sentence, pos_results, chunk_results):
    """Display predictions in a clean table format."""
    print(f'\nInput: "{sentence}"')
    print(f'{"Word":<20} {"POS Tag":<15} {"Chunk Tag"}')
    print('-' * 50)
    for (word, pos), (_, chunk) in zip(pos_results, chunk_results):
        print(f'{word:<20} {pos:<15} {chunk}')


print('✅ Inference function defined.')

In [ ]:
# Test sentences for inference
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Apple announced a new iPhone model yesterday",
    "She is reading an interesting book about machine learning",
    "The president signed the new trade agreement with China"
]

print('=' * 60)
print('INFERENCE — Custom Sentences')
print('=' * 60)

for sentence in test_sentences:
    pos_results   = predict_tags(sentence, pos_model,   tokenizer, pos_id2label)
    chunk_results = predict_tags(sentence, chunk_model, tokenizer, chunk_id2label)
    display_predictions(sentence, pos_results, chunk_results)
    print()


##  Task 7: Comparison — POS Tagging vs Chunking

In [ ]:
# Summary comparison table
comparison_df = pd.DataFrame([
    {
        'Task'          : 'POS Tagging',
        'Level'         : 'Word / Grammar',
        'Label Count'   : len(pos_label_list),
        'Label Format'  : 'NN, VB, JJ, RB...',
        'Difficulty'    : 'Easy',
        'Precision'     : f"{pos_metrics['precision']:.4f}",
        'Recall'        : f"{pos_metrics['recall']:.4f}",
        'F1 Score'      : f"{pos_metrics['f1']:.4f}"
    },
    {
        'Task'          : 'Chunking',
        'Level'         : 'Phrase / Syntactic',
        'Label Count'   : len(chunk_label_list),
        'Label Format'  : 'B-NP, I-NP, B-VP...',
        'Difficulty'    : 'Medium',
        'Precision'     : f"{chunk_metrics['precision']:.4f}",
        'Recall'        : f"{chunk_metrics['recall']:.4f}",
        'F1 Score'      : f"{chunk_metrics['f1']:.4f}"
    }
])

print('=' * 80)
print('COMPARISON TABLE: POS Tagging vs Chunking')
print('=' * 80)
print(comparison_df.to_string(index=False))

In [ ]:
# Visual comparison of label sets
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# POS label example
example_words  = ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog']
example_pos    = ['DT',  'JJ',    'JJ',    'NN',  'VBZ',   'IN',   'DT',  'JJ',   'NN']
example_chunks = ['B-NP','I-NP',  'I-NP',  'I-NP','B-VP',  'B-PP', 'B-NP','I-NP', 'I-NP']

colors_pos   = {'DT': '#93c5fd', 'JJ': '#86efac', 'NN': '#fcd34d', 'VBZ': '#f9a8d4', 'IN': '#c4b5fd'}
colors_chunk = {'B-NP': '#3b82f6', 'I-NP': '#93c5fd', 'B-VP': '#10b981', 'B-PP': '#a855f7'}

for ax, tags, color_map, title in zip(
    axes,
    [example_pos, example_chunks],
    [colors_pos, colors_chunk],
    ['POS Tags — Word Level', 'Chunk Tags — Phrase Level']
):
    for i, (word, tag) in enumerate(zip(example_words, tags)):
        color = color_map.get(tag, '#e5e7eb')
        ax.barh(0, 1, left=i, color=color, edgecolor='white', height=0.5)
        ax.text(i + 0.5, 0,    tag,  ha='center', va='center', fontsize=9,  fontweight='bold')
        ax.text(i + 0.5, -0.35, word, ha='center', va='center', fontsize=8)

    ax.set_xlim(0, len(example_words))
    ax.set_ylim(-0.6, 0.6)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.axis('off')

plt.suptitle('"The quick brown fox jumps over the lazy dog"', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


##  Task 8: Report — Differences, Challenges & Insights

### Differences Between POS Tagging and Chunking

| Aspect | POS Tagging | Chunking |
|---|---|---|
| What it labels | Individual words | Groups of words (phrases) |
| Granularity | Word-level | Phrase-level |
| Labels | NN (noun), VB (verb), JJ (adj)... | B-NP, I-NP, B-VP, O... |
| Label format | Flat tags | IOB (Inside-Outside-Begin) format |
| Difficulty | Easier — each word has one tag | Harder — requires understanding phrase boundaries |
| Use case | Grammar analysis, spell check | Information extraction, parsing |

### The Subword Challenge
The biggest technical challenge in this assignment was **label alignment**. BERT/DistilBERT use WordPiece tokenization which splits words into subword units. For example, `"running"` → `["run", "##ning"]`. Each word has only one label, but the tokenizer produces two tokens for it. We solved this by:
- Assigning the real label to the **first subword token** of each word
- Assigning `-100` to all subsequent subwords (PyTorch ignores `-100` in loss computation)
- Assigning `-100` to special tokens `[CLS]` and `[SEP]`

### Observations
- **POS Tagging achieves higher F1** than Chunking — POS tags are more locally determined (a word's grammatical category is often clear from its form and immediate context), while chunking requires understanding phrase boundaries which is harder.
- **IOB format makes chunking harder** — the model must learn to distinguish B (beginning of a phrase) from I (inside a phrase) in addition to predicting the phrase type.
- **DistilBERT performs well** on both tasks despite being 40% smaller than BERT — proving that for NLP sequence labelling tasks, the distilled model retains most of BERT's capability.
- **Common POS errors** occur on ambiguous words — words that can be both nouns and verbs (e.g. "run", "play") depending on context.
- **Common Chunking errors** occur at phrase boundaries — the model sometimes continues a phrase when a new one should begin (confusing I- with B-).

### Key Takeaways
1. Token classification is fundamentally different from sentence classification — every token needs its own label
2. Label alignment is non-trivial and the most error-prone preprocessing step
3. seqeval is the correct metric for sequence labelling — it evaluates complete spans, not individual tokens
4. The Hugging Face Trainer simplifies training significantly while still giving full control over hyperparameters